# 02 — SigProfiler matrix generation and signature fitting

This notebook:
- generates the SBS96 matrix from the MAF input,
- fits the matrix to COSMIC signatures,
- previews the fitted activity table,
- prints one representative fitted sample with reconstruction quality.


In [ ]:
%matplotlib inline

from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Image, display, Markdown

PROJECT_ROOT = Path.cwd().resolve()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

def show_image(path, width=1000):
    path = Path(path)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Image not found: {path}")

def show_images(*paths, width=1000):
    for path in paths:
        show_image(path, width=width)

def get_sbs_columns(columns):
    cols = [col for col in columns if str(col).startswith("SBS") and str(col)[3:].isdigit()]
    return sorted(cols, key=lambda col: int(str(col)[3:]))

def normalize_rows(frame):
    frame = frame.copy()
    frame = frame.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    row_sums = frame.sum(axis=1)
    frame = frame.loc[row_sums > 0].copy()
    return frame.div(frame.sum(axis=1), axis=0)

def extract_patient_id(value):
    match = re.search(r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})", str(value))
    return match.group(1) if match else np.nan

def map_smoking_label(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"", "nan", "not reported", "unknown"}:
        return "Unknown"
    if "never" in text or "lifelong" in text or "non-smoker" in text or "nonsmoker" in text:
        return "Never"
    if "current reformed" in text or "reformed" in text or "former" in text:
        return "Former"
    if "current" in text:
        return "Current"
    return "Unknown"

def set_spaced_xticks(ax, labels, step=3):
    ax.set_xticks(np.arange(len(labels)) + 0.5)
    ax.set_xticklabels(labels, rotation=90)
    for i, label in enumerate(ax.get_xticklabels()):
        if i % step != 0:
            label.set_visible(False)

    from SigProfilerAssignment import Analyzer as Analyze
    from SigProfilerMatrixGenerator.scripts import SigProfilerMatrixGeneratorFunc


## 1. Define input and output paths


In [ ]:
maf_input_dir = PROJECT_ROOT / "data" / "maf"
matrix_root_dir = maf_input_dir / "output"
signature_fit_dir = PROJECT_ROOT / "results" / "LUAD_sig_output"

signature_fit_dir.mkdir(parents=True, exist_ok=True)

print("MAF input directory :", maf_input_dir)
print("Matrix output root  :", matrix_root_dir)
print("Signature fit output:", signature_fit_dir)


## 2. Preview the MAF input folder


In [ ]:
maf_like_files = sorted(
    [
        path for path in maf_input_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in {".maf", ".txt", ".tsv"}
    ]
)

print(f"Files found: {len(maf_like_files)}")
display(
    pd.DataFrame(
        {
            "file": [path.name for path in maf_like_files[:20]],
            "parent": [str(path.parent.relative_to(PROJECT_ROOT)) for path in maf_like_files[:20]],
        }
    )
)


## 3. Generate the SBS96 matrix


In [ ]:
matrices = SigProfilerMatrixGeneratorFunc.SigProfilerMatrixGeneratorFunc(
    project="LUAD",
    reference_genome="GRCh38",
    path_to_input_files=str(maf_input_dir),
    plot=False,
)

print("SBS96 matrix generation finished.")


## 4. Load and preview the generated matrix


In [ ]:
matrix_candidates = [
    maf_input_dir / "output" / "SBS" / "LUAD.SBS96.all",
    maf_input_dir / "output" / "SBS" / "TCGA_LUAD.SBS96.all",
]

sbs96_path = next((path for path in matrix_candidates if path.exists()), None)

if sbs96_path is None:
    raise FileNotFoundError("The SBS96 matrix file was not found in data/maf/output/SBS/.")

sbs96 = pd.read_csv(sbs96_path, sep="\t", index_col=0)

if sbs96.shape[1] != 96 and sbs96.shape[0] == 96:
    sbs96 = sbs96.T

print("SBS96 matrix path :", sbs96_path)
print("Matrix shape      :", sbs96.shape)
display(sbs96.iloc[:5, :8])


## 5. Fit the SBS96 matrix to COSMIC signatures


In [ ]:
Analyze.cosmic_fit(
    samples=str(sbs96_path),
    output=str(signature_fit_dir),
    input_type="matrix",
    context_type="96",
    collapse_to_SBS96=True,
    cosmic_version=3.4,
)

print("COSMIC fitting finished.")


## 6. Load the fitted activity table


In [ ]:
activities_file = (
    signature_fit_dir
    / "Assignment_Solution"
    / "Activities"
    / "Assignment_Solution_Activities.txt"
)

if not activities_file.exists():
    raise FileNotFoundError(f"Activities file not found: {activities_file}")

activities = pd.read_csv(activities_file, sep="\t", index_col=0)
signature_cols = get_sbs_columns(activities.columns)
activity_matrix = activities[signature_cols].copy()

print("Activities shape :", activity_matrix.shape)
print("Number of fitted SBS signatures:", len(signature_cols))
display(activity_matrix.iloc[:5, :10])

non_zero_signature_totals = activity_matrix.sum(axis=0)
non_zero_signature_totals = non_zero_signature_totals[non_zero_signature_totals > 0].sort_values(ascending=False)
print()
print("Signatures with non-zero total contribution:")
display(non_zero_signature_totals.rename("total_activity").reset_index().rename(columns={"index": "signature"}))


## 7. Print one representative fitted composition


In [ ]:
assignment_root = signature_fit_dir / "Assignment_Solution"
stats_candidates = list(assignment_root.rglob("*Stats*.txt")) + list(assignment_root.rglob("*stats*.txt"))

if not stats_candidates:
    print("No reconstruction statistics file was found automatically.")
    print("Available files near the assignment folder:")
    display(pd.DataFrame({"file": [str(path.relative_to(PROJECT_ROOT)) for path in sorted(assignment_root.rglob('*.txt'))[:30]]}))
else:
    stats_file = stats_candidates[0]
    stats_df = pd.read_csv(stats_file, sep="\t")

    lowered = {col.lower(): col for col in stats_df.columns}

    def find_col(patterns):
        for pattern in patterns:
            for col in stats_df.columns:
                if pattern in col.lower():
                    return col
        return None

    sample_col = find_col(["sample", "samples"])
    cosine_col = find_col(["cosine"])
    l2_col = find_col(["l2"])

    if sample_col is None:
        sample_col = stats_df.columns[0]

    sort_col = cosine_col if cosine_col is not None else (l2_col if l2_col is not None else stats_df.columns[0])
    ascending = False if cosine_col is not None else True

    stats_sorted = stats_df.sort_values(sort_col, ascending=ascending).copy()
    chosen_sample = str(stats_sorted.iloc[0][sample_col])

    activity_row = activity_matrix.loc[chosen_sample].copy()
    active_signatures = activity_row[activity_row > 0].sort_values(ascending=False)

    print("Representative sample:", chosen_sample)
    if cosine_col is not None:
        print("Cosine similarity   :", round(float(stats_sorted.iloc[0][cosine_col]), 4))
    if l2_col is not None:
        print("L2 error            :", round(float(stats_sorted.iloc[0][l2_col]), 4))
    print()
    print("Non-zero fitted signatures:")
    display(
        active_signatures.rename("activity")
        .reset_index()
        .rename(columns={"index": "signature"})
    )
